# Sesión 1 · Del dato al MapReduce
**Profundización I: Procesamiento de Datos — Especialización en Big Data (UCompensar)**

Este notebook acompaña el material HTML de la sesión. Cada celda lleva el mismo
identificador que el bloque de código del material, así que puedes seguir la clase
celda a celda.

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** (Mayúsculas + Enter). Varias usan resultados de las anteriores.
3. Si Colab se reinicia (por ejemplo, tras un break) o una celda falla al repetirla, ve a la celda
   en la que ibas y usa **Entorno de ejecución → Ejecutar anteriores**.
4. La primera celda crea los datos del caso *Tiendas Guadua S.A.S.*, una empresa **ficticia**.

In [ ]:
#@title ▶ Crea los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Tiendas Guadua S.A.S.» (empresa FICTICIA).

Genera los archivos con los que se trabaja durante el curso:

    datos/productos.csv    catálogo de 60 productos
    datos/ventas.csv       ~3 000 ventas del primer semestre de 2025, SUCIAS A PROPÓSITO
    datos/resenas.csv      40 reseñas cortas de clientes (texto libre)
    datos/logs_web.jsonl   eventos de navegación de la tienda en línea (JSON por línea)

Solo usa la biblioteca estándar y una semilla fija: en Colab, en Windows o en
macOS se generan exactamente los mismos archivos. Ninguna cifra del material se
escribe a mano: todas salen de ejecutar el código sobre estos datos.

Uso:  python genera_datos.py [carpeta_destino]      (por defecto ./datos)
"""

import csv
import json
import os
import random
import sys
from datetime import date, datetime, timedelta

SEMILLA = 2026

PRODUCTOS = [
    # (nombre, categoría, precio de lista en pesos)
    ("Arroz blanco 1 kg", "Despensa", 4900),
    ("Lentejas 500 g", "Despensa", 3900),
    ("Fríjol cargamanto 500 g", "Despensa", 6800),
    ("Aceite vegetal 1 L", "Despensa", 11900),
    ("Azúcar 1 kg", "Despensa", 4600),
    ("Sal refinada 1 kg", "Despensa", 1900),
    ("Pasta spaghetti 500 g", "Despensa", 3400),
    ("Harina de maíz 1 kg", "Despensa", 3800),
    ("Atún en lata 170 g", "Despensa", 6900),
    ("Panela 500 g", "Despensa", 3200),
    ("Café molido 500 g", "Despensa", 16900),
    ("Chocolate de mesa 250 g", "Despensa", 7500),
    ("Agua sin gas 600 ml", "Bebidas", 1800),
    ("Gaseosa 1,5 L", "Bebidas", 5200),
    ("Jugo de naranja 1 L", "Bebidas", 6400),
    ("Té helado 500 ml", "Bebidas", 3100),
    ("Bebida energizante 250 ml", "Bebidas", 4900),
    ("Leche entera 1 L", "Bebidas", 4300),
    ("Yogur bebible 1 L", "Bebidas", 8200),
    ("Kumis 1 L", "Bebidas", 7400),
    ("Malta 330 ml", "Bebidas", 2500),
    ("Agua con gas 600 ml", "Bebidas", 2100),
    ("Huevos x 30", "Frescos", 17900),
    ("Pollo entero kg", "Frescos", 13900),
    ("Carne molida 500 g", "Frescos", 12400),
    ("Queso campesino 500 g", "Frescos", 11800),
    ("Plátano verde kg", "Frescos", 3600),
    ("Papa pastusa kg", "Frescos", 2900),
    ("Tomate chonto kg", "Frescos", 4200),
    ("Cebolla cabezona kg", "Frescos", 3300),
    ("Aguacate hass unidad", "Frescos", 3900),
    ("Banano kg", "Frescos", 2800),
    ("Detergente en polvo 1 kg", "Aseo", 14500),
    ("Jabón de loza 500 g", "Aseo", 5900),
    ("Limpiador multiusos 1 L", "Aseo", 7800),
    ("Cloro 1 L", "Aseo", 4100),
    ("Papel higiénico x 12", "Aseo", 21900),
    ("Toallas de cocina x 3", "Aseo", 9600),
    ("Suavizante 1 L", "Aseo", 10900),
    ("Esponjas x 3", "Aseo", 3700),
    ("Bolsas de basura x 10", "Aseo", 6200),
    ("Desinfectante 500 ml", "Aseo", 8900),
    ("Champú 400 ml", "Cuidado personal", 15900),
    ("Crema dental 100 ml", "Cuidado personal", 6300),
    ("Jabón de baño x 3", "Cuidado personal", 8400),
    ("Desodorante 150 ml", "Cuidado personal", 13200),
    ("Cepillo dental", "Cuidado personal", 4800),
    ("Acondicionador 400 ml", "Cuidado personal", 16400),
    ("Protector solar 120 ml", "Cuidado personal", 32900),
    ("Toallas higiénicas x 10", "Cuidado personal", 7900),
    ("Crema corporal 400 ml", "Cuidado personal", 18700),
    ("Bombillo LED 9 W", "Hogar", 7900),
    ("Pilas AA x 4", "Hogar", 12900),
    ("Velas x 6", "Hogar", 5400),
    ("Vasos plásticos x 25", "Hogar", 4300),
    ("Platos desechables x 20", "Hogar", 6900),
    ("Encendedor", "Hogar", 2500),
    ("Extensión eléctrica 3 m", "Hogar", 24900),
    ("Olla 24 cm", "Hogar", 59900),
    ("Juego de cubiertos 16 piezas", "Hogar", 39900),
]

PESO_CATEGORIA = {"Despensa": 1.6, "Bebidas": 1.5, "Frescos": 1.4, "Aseo": 1.0,
                  "Cuidado personal": 0.8, "Hogar": 0.5}

CIUDADES = ["Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Cartagena"]
PESO_CIUDAD = [0.34, 0.20, 0.15, 0.12, 0.10, 0.09]

# Formas "sucias" en que llega escrita cada ciudad desde los distintos sistemas
VARIANTES_CIUDAD = {
    "Bogotá": ["bogota", "BOGOTÁ", "Bogotá D.C.", "Bogota", " Bogotá"],
    "Medellín": ["Medellin", "medellín", "MEDELLIN"],
    "Cali": ["cali", "Cali ", "CALI"],
    "Barranquilla": ["barranquilla", "B/quilla", "Barranquilla "],
    "Bucaramanga": ["bucaramanga", "B/manga"],
    "Cartagena": ["cartagena", "Cartagena de Indias"],
}

VARIANTES_CANAL = {"Tienda": ["tienda", "TIENDA"], "Web": ["WEB", "web "], "App": ["app", "APP"]}

PAGO_POR_CANAL = {
    "Tienda": (["Efectivo", "Tarjeta", "Billetera digital"], [0.35, 0.45, 0.20]),
    "Web": (["Tarjeta", "Transferencia", "Billetera digital"], [0.60, 0.30, 0.10]),
    "App": (["Billetera digital", "Tarjeta", "Transferencia"], [0.50, 0.40, 0.10]),
}

RESENAS = [
    # (calificación, texto). Reseñas ficticias escritas para el curso.
    (5, "La entrega fue muy rápida y el pedido llegó completo. Excelente servicio."),
    (4, "Buenos precios y buena calidad, pero la app se demora en cargar el carrito."),
    (2, "El pedido llegó tarde y faltaba un producto. Tuve que llamar dos veces."),
    (5, "Me encanta comprar por la app, es fácil y el domicilio siempre llega a tiempo."),
    (1, "Pésima experiencia: cobraron dos veces el pago y nadie responde en el chat."),
    (4, "Los frescos llegaron en buen estado. El aguacate estaba perfecto."),
    (3, "Precios normales. La tienda es amplia pero las filas en caja son largas."),
    (5, "Excelente atención en la tienda de Medellín, el personal es muy amable."),
    (2, "La página web se cae al momento de pagar. Perdí el carrito dos veces."),
    (4, "Buena variedad de productos de aseo y descuentos interesantes en la app."),
    (1, "Llegó el huevo roto y el pollo sin refrigerar. No vuelvo a pedir frescos."),
    (5, "Rápido, económico y sin complicaciones. Recomendado."),
    (3, "El domicilio llegó bien, pero el empaque venía golpeado."),
    (4, "Me gusta pagar con billetera digital, es rápido. Faltan más promociones."),
    (2, "El precio en la app no coincidía con el precio en la tienda."),
    (5, "Pedí a las ocho de la mañana y a las diez ya tenía todo en casa. Muy bien."),
    (3, "Regular. La calidad del café ha bajado y el precio subió."),
    (4, "La tienda de Cali siempre está limpia y ordenada. Buen surtido."),
    (1, "Cancelaron mi pedido sin avisar y el reembolso tardó una semana."),
    (5, "Excelente calidad en carnes y quesos. El domicilio fue puntual."),
    (2, "La app no me deja aplicar el cupón de descuento. Muy frustrante."),
    (4, "Buena experiencia en general, aunque el pedido llegó un poco tarde."),
    (3, "Productos buenos pero la entrega es lenta los fines de semana."),
    (5, "Siempre encuentro lo que necesito y a buen precio. La mejor opción del barrio."),
    (1, "El servicio al cliente es terrible, nadie resuelve nada por el chat."),
    (4, "La compra por la web fue fácil y el pago con tarjeta funcionó sin problema."),
    (2, "El producto llegó vencido. Revisen las fechas antes de enviar."),
    (5, "Muy buena atención en Barranquilla, rápidos en caja y muy amables."),
    (3, "Normal. Nada especial, pero cumple."),
    (4, "Las ofertas de los martes son muy buenas, compro el mercado completo."),
    (2, "Demasiada demora en el domicilio y el repartidor no encontraba la dirección."),
    (5, "Calidad excelente y precios justos. La app funciona perfecto."),
    (1, "Me llegó un pedido equivocado y no me dejaron devolverlo."),
    (4, "Buen precio en productos de despensa. El arroz y el aceite siempre en oferta."),
    (3, "La tienda de Bucaramanga es pequeña y a veces no hay todo lo del catálogo."),
    (5, "Entrega rápida, todo bien empacado y frío. Muy recomendado."),
    (2, "Cobran el domicilio muy caro para pedidos pequeños."),
    (4, "Me gusta que en la app se ve el estado del pedido en tiempo real."),
    (3, "El pago con transferencia tardó en confirmarse, pero al final llegó todo."),
    (5, "Excelente servicio y excelente calidad. Seguiré comprando en Cartagena."),
]


def _semestre(rng):
    """Días del 1 de enero al 30 de junio de 2025, con más peso el fin de semana
    y una tendencia suave al alza."""
    inicio = date(2025, 1, 1)
    dias = [inicio + timedelta(days=i) for i in range((date(2025, 6, 30) - inicio).days + 1)]
    pesos = [(1.4 if d.weekday() >= 5 else 1.0) * (1 + 0.12 * (d.month - 1) / 5) for d in dias]
    return dias, pesos


def _ventas(rng):
    dias, pesos_dia = _semestre(rng)
    pesos_producto = [PESO_CATEGORIA[c] for _, c, _ in PRODUCTOS]
    clientes = [f"C{i:04d}" for i in range(1, 1501)]
    # Pocos clientes compran mucho: pesos decrecientes
    pesos_cliente = [1 / (1 + i) ** 0.6 for i in range(len(clientes))]

    filas = []
    for dia in sorted(rng.choices(dias, weights=pesos_dia, k=3000)):
        ciudad = rng.choices(CIUDADES, weights=PESO_CIUDAD)[0]
        # La app gana participación mes a mes
        p_app = 0.12 + 0.03 * (dia.month - 1)
        canal = rng.choices(["Tienda", "Web", "App"], weights=[0.63 - p_app, 0.25, p_app + 0.12])[0]
        indice = rng.choices(range(len(PRODUCTOS)), weights=pesos_producto)[0]
        _, _, precio = PRODUCTOS[indice]
        if precio < 10000:
            cantidad = rng.choices([1, 2, 3, 4, 5, 6], weights=[30, 28, 18, 12, 7, 5])[0]
        else:
            cantidad = rng.choices([1, 2, 3], weights=[70, 24, 6])[0]
        if canal == "Tienda":
            descuento = rng.choices([0, 0.05, 0.10], weights=[85, 10, 5])[0]
        else:
            descuento = rng.choices([0, 0.05, 0.10, 0.15], weights=[55, 20, 15, 10])[0]
        medios, pesos_pago = PAGO_POR_CANAL[canal]
        pago = rng.choices(medios, weights=pesos_pago)[0]
        # En tienda se puede comprar sin registrarse: ese nulo NO es un error
        if canal == "Tienda" and rng.random() < 0.40:
            cliente = ""
        else:
            cliente = rng.choices(clientes, weights=pesos_cliente)[0]
        filas.append({
            "fecha": dia.isoformat(),
            "ciudad": ciudad,
            "canal": canal,
            "id_cliente": cliente,
            "id_producto": f"P{indice + 1:03d}",
            "cantidad": cantidad,
            "precio_unitario": precio,
            "descuento": descuento,
            "metodo_pago": pago,
        })

    for i, f in enumerate(filas, start=1):
        f["id_venta"] = f"V{i:05d}"

    # ---- Suciedad controlada: los problemas típicos de un consolidado real ----
    for f in filas:
        r = rng.random()
        if r < 0.07:
            f["ciudad"] = rng.choice(VARIANTES_CIUDAD[f["ciudad"]])
        elif r < 0.075:
            f["ciudad"] = ""                                   # ciudad sin registrar
        if rng.random() < 0.03:
            f["canal"] = rng.choice(VARIANTES_CANAL[f["canal"]])
        if rng.random() < 0.025:
            f["precio_unitario"] = "$" + f"{f['precio_unitario']:,}".replace(",", ".")
        if rng.random() < 0.04:
            d = date.fromisoformat(f["fecha"])
            f["fecha"] = d.strftime("%d/%m/%Y")                # otro formato de fecha
        r = rng.random()
        if r < 0.004:
            f["cantidad"] = -rng.randint(1, 3)                 # cantidad negativa
        elif r < 0.006:
            f["cantidad"] = 0
        elif r < 0.0075:
            f["cantidad"] = f["cantidad"] * 100 + rng.randint(0, 9) * 10   # error de digitación
        if rng.random() < 0.02:
            f["descuento"] = ""
        if rng.random() < 0.01:
            f["metodo_pago"] = ""

    # Doble carga: ~1 % de las ventas aparece repetida justo después del original
    con_duplicados = []
    for f in filas:
        con_duplicados.append(f)
        if rng.random() < 0.01:
            con_duplicados.append(dict(f))
    return con_duplicados


def _resenas(rng):
    dias, _ = _semestre(rng)
    filas = []
    for i, (calificacion, texto) in enumerate(RESENAS, start=1):
        fila = {
            "id_resena": f"R{i:03d}",
            "fecha": rng.choice(dias).isoformat(),
            "ciudad": rng.choices(CIUDADES, weights=PESO_CIUDAD)[0],
            "canal": rng.choices(["Tienda", "Web", "App"], weights=[0.3, 0.3, 0.4])[0],
            "id_producto": f"P{rng.randint(1, len(PRODUCTOS)):03d}",
            "calificacion": calificacion,
            "texto": texto,
        }
        # Lo que dice el texto manda sobre el sorteo (sin más llamadas a rng: ventas y logs no cambian)
        t = texto.lower()
        for ciudad in CIUDADES:
            if ciudad.lower() in t:
                fila["ciudad"] = ciudad
        if " app" in t:
            fila["canal"] = "App"
        elif "web" in t:
            fila["canal"] = "Web"
        elif "tienda de" in t or "en caja" in t or "la tienda es" in t:
            fila["canal"] = "Tienda"
        elif fila["canal"] == "Tienda" and any(p in t for p in ("domicilio", "pedido", "pedí", "entrega",
                                                                "repartidor", "llegó", "envi", "chat")):
            fila["canal"] = "Web"                      # un domicilio o un chat no son de la caja
        filas.append(fila)
    filas.sort(key=lambda f: f["fecha"])
    return filas


def _logs(rng, n=1500):
    """Eventos de la tienda en línea en junio de 2025. Semiestructurados: no
    todos los eventos tienen los mismos campos y algunos van anidados."""
    eventos = []
    inicio = datetime(2025, 6, 1)
    for _ in range(n):
        ts = inicio + timedelta(seconds=rng.randint(0, 30 * 24 * 3600 - 1))
        tipo = rng.choices(["buscar", "ver_producto", "agregar_carrito", "pagar"],
                           weights=[25, 45, 20, 10])[0]
        movil = rng.random() < 0.62
        evento = {
            "ts": ts.isoformat(timespec="seconds"),
            "sesion": f"s-{rng.randint(0, 0xFFFFF):05x}",
            "evento": tipo,
            "dispositivo": {"tipo": "movil" if movil else "escritorio",
                            "so": rng.choice(["android", "ios"]) if movil else rng.choice(["windows", "macos", "linux"])},
            "ciudad": rng.choices(CIUDADES, weights=PESO_CIUDAD)[0],
        }
        if rng.random() < 0.7:
            evento["id_cliente"] = f"C{rng.randint(1, 1500):04d}"
        if tipo == "buscar":
            evento["consulta"] = rng.choice(["arroz", "cafe", "detergente", "leche", "huevos",
                                             "protector solar", "pilas", "pollo", "aceite"])
        else:
            evento["id_producto"] = f"P{rng.randint(1, len(PRODUCTOS)):03d}"
        if tipo == "pagar":
            evento["pago"] = {"medio": rng.choice(["Tarjeta", "Billetera digital", "Transferencia"]),
                              "aprobado": rng.random() < 0.9}
        evento["duracion_ms"] = rng.randint(80, 4000)
        eventos.append(evento)
    eventos.sort(key=lambda e: e["ts"])
    return eventos


def _escribir_csv(ruta, filas, columnas):
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=columnas)
        escritor.writeheader()
        escritor.writerows(filas)


def crear_datos(carpeta="datos"):
    """Crea los cuatro archivos del caso en `carpeta` y devuelve sus rutas."""
    os.makedirs(carpeta, exist_ok=True)
    rng = random.Random(SEMILLA)

    productos = [{"id_producto": f"P{i:03d}", "nombre": n, "categoria": c, "precio_lista": p}
                 for i, (n, c, p) in enumerate(PRODUCTOS, start=1)]
    ventas = _ventas(rng)
    resenas = _resenas(rng)
    logs = _logs(rng)

    rutas = {
        "productos": os.path.join(carpeta, "productos.csv"),
        "ventas": os.path.join(carpeta, "ventas.csv"),
        "resenas": os.path.join(carpeta, "resenas.csv"),
        "logs": os.path.join(carpeta, "logs_web.jsonl"),
    }
    _escribir_csv(rutas["productos"], productos, ["id_producto", "nombre", "categoria", "precio_lista"])
    _escribir_csv(rutas["ventas"], ventas,
                  ["id_venta", "fecha", "ciudad", "canal", "id_cliente", "id_producto",
                   "cantidad", "precio_unitario", "descuento", "metodo_pago"])
    _escribir_csv(rutas["resenas"], resenas,
                  ["id_resena", "fecha", "ciudad", "canal", "id_producto", "calificacion", "texto"])
    with open(rutas["logs"], "w", encoding="utf-8") as f:
        for e in logs:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

    print(f"productos.csv : {len(productos):>5} filas")
    print(f"ventas.csv    : {len(ventas):>5} filas")
    print(f"resenas.csv   : {len(resenas):>5} filas")
    print(f"logs_web.jsonl: {len(logs):>5} eventos")
    return rutas

# Crea la carpeta datos/ con los cuatro archivos del caso
rutas = crear_datos("datos")

## Bloque 1 · Módulo 5 — Primer contacto con el entorno

In [ ]:
# [entorno]
import sys
import pandas as pd

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

In [ ]:
# [crudo]
# Las primeras líneas del archivo, tal como están en el disco
!head -n 4 datos/ventas.csv

## Bloque 3 · Módulo 10 — Python para datos en 10 minutos

In [ ]:
# [py_basico]
# Listas: colecciones ordenadas
ciudades = ["Bogotá", "Medellín", "Cali"]
montos = [120_000, 85_500, 64_200]
print(len(ciudades), ciudades[0], sum(montos))

# Diccionarios: pares clave → valor
venta = {"id_venta": "V00001", "ciudad": "Bogotá", "total": 45_900}
print(venta["ciudad"])

# Funciones: lógica reutilizable con nombre
def con_iva(valor, tasa=0.19):
    return round(valor * (1 + tasa))

print(con_iva(10_000))

# lambda (función anónima) y comprensión de listas
en_miles = lambda x: x / 1000
print([en_miles(m) for m in montos])
print([c.upper() for c in ciudades if c != "Cali"])

In [ ]:
# [py_pandas]
import pandas as pd

# Un DataFrame es una tabla: columnas con nombre, todas del mismo largo
mini = pd.DataFrame({"ciudad": ciudades, "monto": montos})
print(mini)
print("Promedio:", mini["monto"].mean())

# Filtrar: la condición da True/False por fila (una «máscara») y se quedan las True
print(mini[mini["monto"] > 80_000])

## Bloque 3 · Módulo 11 — Cargar e inspeccionar

In [ ]:
# [cargar]
ventas = pd.read_csv("datos/ventas.csv")
productos = pd.read_csv("datos/productos.csv")
print("ventas:", ventas.shape, "· productos:", productos.shape)
ventas.head()

In [ ]:
# [info]
ventas.info()

In [ ]:
# [describe]
ventas.describe()

In [ ]:
# [conteos]
ventas["ciudad"].value_counts()

In [ ]:
# [nulos]
ventas.isna().sum()

**Ejercicio (módulo 11).** ¿Cuántos clientes distintos hay en el archivo y qué porcentaje de las
ventas no tiene cliente? Resuélvelo en una celda nueva (**+ Código**); la pista y la solución están en el material HTML.

## Bloque 3 · Módulo 12 — Limpieza y calidad del dato
Cada paso de limpieza responde a una **dimensión de calidad**. Trabajamos sobre una
copia (`df`) para poder comparar el antes y el después.

In [ ]:
# [duplicados]
# UNICIDAD — la misma venta cargada dos veces
df = ventas.copy()            # el original no se toca
print("Filas iniciales:", len(df))
print("Filas duplicadas:", df.duplicated().sum())
df = df.drop_duplicates()
print("Filas tras quitar duplicados:", len(df))

In [ ]:
# [texto]
# CONSISTENCIA — una misma ciudad escrita de muchas formas
def normalizar(serie):
    """Minúsculas, sin espacios sobrantes y sin tildes."""
    return (serie.str.strip()
                 .str.lower()
                 .str.normalize("NFKD")
                 .str.encode("ascii", errors="ignore")
                 .str.decode("ascii"))

CIUDAD_CANONICA = {
    "bogota": "Bogotá", "bogota d.c.": "Bogotá",
    "medellin": "Medellín",
    "cali": "Cali",
    "barranquilla": "Barranquilla", "b/quilla": "Barranquilla",
    "bucaramanga": "Bucaramanga", "b/manga": "Bucaramanga",
    "cartagena": "Cartagena", "cartagena de indias": "Cartagena",
}

ciudad_norm = normalizar(df["ciudad"])
sin_regla = ciudad_norm[ciudad_norm.notna() & ~ciudad_norm.isin(CIUDAD_CANONICA)]
print("Grafías sin regla:", sin_regla.unique())        # debe salir vacío

df["ciudad"] = ciudad_norm.map(CIUDAD_CANONICA).fillna("Sin dato")
df["canal"] = df["canal"].str.strip().str.capitalize()
print(df["ciudad"].value_counts().to_dict())
print(df["canal"].value_counts().to_dict())

In [ ]:
# [precios]
# VALIDEZ — precios guardados como texto con «$» y separador de miles
precio_txt = df["precio_unitario"].astype(str)
con_signo = precio_txt.str.contains("$", regex=False)
print("Precios con «$» y punto de miles:", con_signo.sum())
print(precio_txt[con_signo].head(3).tolist())

# Los precios en pesos son enteros: se quitan «$», puntos y espacios
df["precio_unitario"] = pd.to_numeric(
    precio_txt.str.replace(r"[$.\s]", "", regex=True), errors="coerce")
print("Precios sin convertir:", df["precio_unitario"].isna().sum())
print("Tipo final:", df["precio_unitario"].dtype)

In [ ]:
# [fechas]
# VALIDEZ — fechas en dos formatos: se interpreta cada formato de forma EXPLÍCITA
iso = pd.to_datetime(df["fecha"], format="%Y-%m-%d", errors="coerce")
dma = pd.to_datetime(df["fecha"], format="%d/%m/%Y", errors="coerce")
print("Fechas en formato dd/mm/aaaa:", (iso.isna() & dma.notna()).sum())

df["fecha"] = iso.fillna(dma)
print("Fechas sin interpretar:", df["fecha"].isna().sum())
print("Rango:", df["fecha"].min().date(), "a", df["fecha"].max().date())

In [ ]:
# [reglas]
# EXACTITUD — cantidades imposibles (negativas, cero o errores de digitación)
fuera_de_rango = ~df["cantidad"].between(1, 50)
print("Cantidades fuera de [1, 50]:", fuera_de_rango.sum())
print(df.loc[fuera_de_rango, "cantidad"].value_counts().sort_index().to_dict())

cuarentena = df[fuera_de_rango].copy()  # no se borran sin dejar rastro: se apartan
df = df[~fuera_de_rango].copy()
print("Filas válidas:", len(df), "· en cuarentena:", len(cuarentena))

In [ ]:
# [completitud]
# COMPLETITUD — cada vacío se trata con una regla de negocio DOCUMENTADA
df["descuento"] = df["descuento"].fillna(0)               # sin registro = sin descuento
df["id_cliente"] = df["id_cliente"].fillna("ANONIMO")     # compra en tienda sin registro
df["metodo_pago"] = df["metodo_pago"].fillna("Sin dato")  # no se inventa un medio de pago
df.isna().sum()

In [ ]:
# [reporte]
# Los mismos indicadores sobre el original (antes) y sobre la tabla limpia (después)
def reporte_calidad(d):
    """Indicadores de calidad de una tabla de ventas."""
    precio = pd.to_numeric(d["precio_unitario"], errors="coerce")
    fecha = pd.to_datetime(d["fecha"], format="%Y-%m-%d", errors="coerce")
    return pd.Series({
        "filas": len(d),
        "filas duplicadas": d.duplicated().sum(),
        "celdas vacías": d.isna().sum().sum(),
        "grafías de ciudad": d["ciudad"].nunique(),
        "grafías de canal": d["canal"].nunique(),
        "precios no numéricos": precio.isna().sum(),
        "fechas sin interpretar": fecha.isna().sum(),
        "cantidades fuera de rango": (~d["cantidad"].between(1, 50)).sum(),
    })

pd.DataFrame({"antes": reporte_calidad(ventas), "después": reporte_calidad(df)})

**Ejercicio (módulo 12).** Los vacíos de `id_cliente`, ¿son un error de calidad? Averígualo contando
cuántos hay en cada canal de `ventas`. Resuélvelo en una celda nueva (**+ Código**); la pista y la solución están en el material HTML.

## Bloque 3 · Módulo 13 — Transformar y resumir

In [ ]:
# [derivadas]
# Columnas derivadas: el total de cada venta y dos llaves de tiempo
df["total"] = df["cantidad"] * df["precio_unitario"] * (1 - df["descuento"])
df["mes"] = df["fecha"].dt.strftime("%Y-%m")
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
df["dia_semana"] = df["fecha"].dt.dayofweek.map(dict(enumerate(DIAS)))
df[["id_venta", "cantidad", "precio_unitario", "descuento", "total", "mes", "dia_semana"]].head()

In [ ]:
# [union]
# Unir con el catálogo para traer el nombre y la categoría de cada producto
df = (df.drop(columns=["nombre", "categoria"], errors="ignore")   # repetible sin duplicar columnas
        .merge(productos[["id_producto", "nombre", "categoria"]],
               on="id_producto", how="left", validate="many_to_one"))
print("Ventas sin producto en el catálogo:", df["categoria"].isna().sum())
df[["id_venta", "nombre", "categoria", "total"]].head(3)

In [ ]:
# [por_ciudad]
# Dividir – aplicar – combinar: un resumen por ciudad
por_ciudad = (df.groupby("ciudad")
                .agg(ventas=("id_venta", "count"),
                     total_millones=("total", lambda s: s.sum() / 1e6),
                     ticket_promedio=("total", "mean"))
                .sort_values("total_millones", ascending=False)
                .round({"total_millones": 1, "ticket_promedio": 0}))
por_ciudad

In [ ]:
# [pivote]
# Tabla dinámica: ventas (millones de pesos) por categoría y canal
tabla = pd.pivot_table(df, index="categoria", columns="canal", values="total",
                       aggfunc="sum", margins=True, margins_name="Total") / 1e6
tabla.round(1)

In [ ]:
# [app_mes]
# ¿Qué porcentaje de las ventas de cada mes entra por cada canal?
participacion = pd.crosstab(df["mes"], df["canal"], values=df["total"],
                            aggfunc="sum", normalize="index") * 100
participacion.round(1)

**Ejercicios (módulo 13).**
1. ¿Qué día de la semana tiene el mayor ticket promedio (total promedio por venta)?
2. ¿Qué categoría vende más por la app, en millones de pesos?

Resuélvelo en una celda nueva (**+ Código**); la pista y la solución están en el material HTML.

## Bloque 4 · Módulo 14 — Pensar en map, filter y reduce

In [ ]:
# [bucle_for]
# Forma 1 · bucle for: dice CÓMO, paso a paso
precios = [12_900, 4_500, 23_800, 8_700]

con_impuesto = []
for p in precios:
    con_impuesto.append(round(p * 1.19))
print(con_impuesto)

In [ ]:
# [con_map]
# Forma 2 · map + lambda: dice QUÉ aplicar a cada elemento
precios = [12_900, 4_500, 23_800, 8_700]

con_impuesto = list(map(lambda p: round(p * 1.19), precios))
print(con_impuesto)

In [ ]:
# [comprension]
# Forma 3 · comprensión de listas: lo mismo, en la forma más común en Python
precios = [12_900, 4_500, 23_800, 8_700]

con_impuesto = [round(p * 1.19) for p in precios]
print(con_impuesto)

In [ ]:
# [filter_reduce]
from functools import reduce

caros = list(filter(lambda p: p > 10_000, precios))     # se quedan los que cumplen
suma = reduce(lambda acumulado, p: acumulado + p, precios, 0)
print(caros)
print(suma, "==", sum(precios))

## Bloque 4 · Módulo 15 — MapReduce a mano: conteo de palabras

In [ ]:
# [wc_textos]
resenas = pd.read_csv("datos/resenas.csv")
textos = resenas["texto"].tolist()
print(len(textos), "reseñas")
print(textos[0])

In [ ]:
# [wc_map]
import re

def mapper(texto):
    """Fase MAP: de un texto a una lista de pares (palabra, 1)."""
    palabras = re.findall(r"[a-záéíóúüñ]+", texto.lower())
    return [(palabra, 1) for palabra in palabras]

print(mapper(textos[0]))
pares = [par for texto in textos for par in mapper(texto)]
print("Pares emitidos:", len(pares))

In [ ]:
# [wc_shuffle]
from collections import defaultdict

# Fase SHUFFLE: agrupar los valores de cada clave; sorted imita el orden por clave que garantiza Hadoop
grupos = defaultdict(list)
for palabra, uno in sorted(pares):
    grupos[palabra].append(uno)
print("Palabras distintas:", len(grupos))
print("entrega →", grupos["entrega"])

In [ ]:
# [wc_reduce]
def reducer(palabra, unos):
    """Fase REDUCE: combinar todos los valores de una misma clave."""
    return palabra, sum(unos)

conteo = [reducer(p, v) for p, v in grupos.items()]
conteo.sort(key=lambda par: par[1], reverse=True)
conteo[:10]

In [ ]:
# [wc_vacias]
# Las palabras vacías (artículos, preposiciones…) no dicen nada del negocio
VACIAS = {"el", "la", "los", "las", "un", "una", "y", "o", "de", "del", "a", "al",
          "en", "con", "por", "para", "que", "es", "muy", "se", "me", "mi", "no",
          "lo", "le", "pero", "más", "fue", "su", "sin", "todo", "siempre"}
[(p, n) for p, n in conteo if p not in VACIAS][:10]

In [ ]:
# [wc_counter]
from collections import Counter

# La biblioteca estándar ya trae este conteo: debe coincidir con el nuestro
dict(Counter(p for p, _ in pares)) == dict(conteo)

**Ejercicio (módulo 15).** Reutiliza `mapper` para contar las palabras (sin vacías) de las reseñas con
calificación de 2 o menos. ¿De qué se quejan los clientes? Resuélvelo en una celda nueva (**+ Código**); la pista y la solución están en el material HTML.

## Bloque 4 · Módulo 16 — MapReduce sobre las ventas y estilo Hadoop Streaming

In [ ]:
# [mr_ventas]
# 1) Partir los datos en 4 bloques, como HDFS parte un archivo grande
tam = -(-len(df) // 4)                          # división hacia arriba
bloques = [df.iloc[i:i + tam] for i in range(0, len(df), tam)]
print("Filas por bloque:", [len(b) for b in bloques])

# 2) MAP + COMBINER: cada bloque emite (ciudad, total) y suma localmente
def map_y_combinar(bloque):
    parcial = defaultdict(float)
    for ciudad, total in zip(bloque["ciudad"], bloque["total"]):
        parcial[ciudad] += total
    return parcial

parciales = [map_y_combinar(b) for b in bloques]   # en un clúster: en paralelo

# 3) SHUFFLE + REDUCE: sumar los parciales de cada ciudad
resultado = defaultdict(float)
for parcial in parciales:
    for ciudad, subtotal in parcial.items():
        resultado[ciudad] += subtotal

mr = pd.Series(resultado).sort_values(ascending=False) / 1e6
mr.round(1)

In [ ]:
# [mr_verifica]
# Control de coherencia: MapReduce y groupby deben dar lo mismo
con_groupby = df.groupby("ciudad")["total"].sum() / 1e6
diferencia = (mr - con_groupby).abs().max()
print("Máxima diferencia:", diferencia)
print("¿Coinciden?", diferencia < 1e-9)

**Ejercicio (módulo 16).** Con los mismos cuatro `bloques`, cuenta con MapReduce cuántas ventas hubo
por método de pago y verifícalo contra `value_counts()`. Resuélvelo en una celda nueva (**+ Código**); la pista y la solución están en el material HTML.

In [ ]:
# [exporta]
# Hadoop Streaming lee y escribe TEXTO: exportamos solo lo que necesita el job
df[["ciudad", "total"]].to_csv("ventas_limpias.csv", index=False)
!head -n 3 ventas_limpias.csv

In [ ]:
%%writefile mapper.py
import sys

# MAPPER: lee líneas de la entrada estándar y emite «clave<TAB>valor»
for linea in sys.stdin:
    ciudad, total = linea.strip().split(",")
    if ciudad == "ciudad":        # salta el encabezado
        continue
    print(f"{ciudad}\t{total}")

In [ ]:
%%writefile reducer.py
import sys

# REDUCER: recibe las líneas ORDENADAS por clave y acumula mientras la clave no cambie
actual, suma = None, 0.0
for linea in sys.stdin:
    clave, valor = linea.rstrip("\n").split("\t")
    if clave != actual:
        if actual is not None:
            print(f"{actual}\t{suma:.0f}")
        actual, suma = clave, 0.0
    suma += float(valor)
if actual is not None:
    print(f"{actual}\t{suma:.0f}")

In [ ]:
# [streaming]
# mapper → sort (el «shuffle» de Hadoop) → reducer, con tuberías de Unix
!cat ventas_limpias.csv | python3 mapper.py | sort | python3 reducer.py

## Para el taller
El enunciado completo y la rúbrica están en el módulo 17 del material HTML.
Sigue trabajando en este mismo notebook: ya tienes `df` limpio, las funciones `normalizar`,
`reporte_calidad`, `mapper` y `reducer` (conteo de palabras) y los archivos `mapper.py` y
`reducer.py` de Streaming.